In [ ]:
import os, pickle, socket, struct, threading, time
import pygame

from netcode import NetClient

pygame.init()


In [ ]:
screen = pygame.display.set_mode((1920, 1080))
backgroundImage = pygame.image.load('blueSky.jpg').convert()
backgroundImage = pygame.transform.scale(backgroundImage, (1920, 1080))
terrain  = pygame.image.load(("terrainSheet.png")).convert_alpha()
platform = terrain.subsurface(pygame.Rect(272, 16, 48, 16))
platform = pygame.transform.scale_by(platform, 3)
player_image = pygame.image.load("frog.png").convert_alpha()
player_image = pygame.transform.scale_by(player_image, 3)
world_tileset = pygame.image.load("world_tileset.png").convert_alpha()
ground_tile = world_tileset.subsurface(pygame.Rect(0, 0, 16, 16))
ground_tile = pygame.transform.scale_by(ground_tile, 3)





In [ ]:
PLATFORM_WIDTH, PLATFORM_HEIGHT = platform.get_size()

PLATFORMS = [
    (500, 950),
    (900, 800),
    (1300, 650),
    (1200, 475),
    (900, 400),
    (700, -350),
    (300, -200),
    (1050, -200),
    (650, -50),
    (250, 100),
    (1000, 100),
    (550, 250),
    (1300, 250)
]


In [ ]:
GROUND_WIDTH, GROUND_HEIGHT = ground_tile.get_size()

GROUND = [
    (x, 1080 - GROUND_HEIGHT) for x in range(0, 1920, GROUND_WIDTH)
]

In [ ]:
#platform_rects = [pygame.Rect(x, y, PLATFORM_WIDTH, PLATFORM_HEIGHT) for x, y in PLATFORMS]
#ground_rects = [pygame.Rect(x, y, GROUND_WIDTH, GROUND_HEIGHT) for x, y in GROUND]
#collidable_rects = platform_rects + ground_rects

In [ ]:
_tint_cache = {}

def tinted_frog(tint):
    """The frog sprite multiplied by a colour, so players can be told apart.

    Needs an alpha surface, which player_image is (loaded with .convert_alpha()).
    Cached because it runs once per remote player per frame otherwise.
    """
    key = tuple(tint)
    if key not in _tint_cache:
        img = player_image.copy()
        img.fill((tint[0], tint[1], tint[2], 255),
                 special_flags=pygame.BLEND_RGBA_MULT)
        _tint_cache[key] = img
    return _tint_cache[key]


In [ ]:
class Camera:

    def __init__(self):
        self.offset_x = 0
        self.offset_y = 0
        self.move_speed = 5

    def follow(self, player):
        self.offset_x = screen.get_width() // 2 - player.x
        self.offset_y = screen.get_height() // 2 - player.y

    def render_world(self, platforms, ground, remotes=()):
        self.follow(player)

        screen.fill((0, 0, 0))
        screen.blit(backgroundImage, (0, 0))

        for x, y in platforms:
            screen.blit(platform, (x+self.offset_x, y+self.offset_y))

        for x, y in ground:
            screen.blit(ground_tile, (x+self.offset_x, y+self.offset_y))

       
        for p in remotes:
            screen.blit(tinted_frog(p.tint),
                        (p.x + self.offset_x, p.y + self.offset_y))

        screen.blit(player.image, (player.x + self.offset_x, player.y + self.offset_y))

In [ ]:
class Player:
  def __init__(self, image, camera: Camera):
      #Koordinaterna
      self.x = 300
      self.y = 300
      self.speed = 5
      self.vel_y = 0
      self.gravity = 0.5
      self.jump_power = -10
      self.on_ground = False
      self.image = image
      self.camera = camera

  def move(self, left, right, jump):
    self.speed = 5
    
    if left:
      self.x -= self.speed

    if right:
      self.x += self.speed

    if jump and self.on_ground:
        self.vel_y = self.jump_power
        self.on_ground = False

  def apply_gravity(self):
      if self.on_ground == False:
        self.vel_y += self.gravity
        self.y += self.vel_y
        if (self.vel_y >=10):
                  self.vel_y = 10
        
      
      if is_Grounded(self):
          self.vel_y = 0
          self.on_ground = True
      else:
          self.on_ground = False


In [ ]:
def handle_input():
  keys = pygame.key.get_pressed()

  left = keys[pygame.K_a]
  right = keys[pygame.K_d]
  jump = keys[pygame.K_SPACE]

  return left, right, jump

In [ ]:
GROUND_TOLERANCE = 12

def is_Grounded(player: Player):
    player_left = player.x
    player_right = player.x + player.image.get_width()
    player_bottom = player.y + player.image.get_height()

    for x, y in PLATFORMS:
        landed_on_top = abs(player_bottom - y) <= GROUND_TOLERANCE
        within_platform_x = player_right > x and player_left < x + PLATFORM_WIDTH
        if landed_on_top and within_platform_x:
            return True

    for x, y in GROUND:
        landed_on_top = abs(player_bottom - y) <= GROUND_TOLERANCE
        within_ground_x = player_right > x and player_left < x + GROUND_WIDTH
        if landed_on_top and within_ground_x:
            return True

    return False

In [ ]:
PLAYER_NAME = "david"          # your save-file key -- everyone picks a different one
SERVER_HOST = "127.0.0.1"      # the machine running server.py

try:                           # re-running this cell must not leak a second connection
    net.close()
except NameError:
    pass

net = NetClient(SERVER_HOST, PLAYER_NAME)
print("connected as player", net.my_id, "| tint", net.tint, "| resume", net.resume)


In [ ]:

camera = Camera()
player = Player(player_image, camera)

# Restore a saved world position. player.x/y ARE world coordinates now, so
# resuming just means placing the player back where it left off.
if net.resume is not None:
    player.x, player.y = net.resume

clock = pygame.time.Clock()


running = True
while running:
    for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
    
    left, right, jump = handle_input()
    
    player.move(left,right,jump)
    player.apply_gravity()

    
    remotes = net.update(player.x, player.y)

    camera.render_world(PLATFORMS, GROUND, remotes)

    status = "online" if net.connected else "OFFLINE"
    pygame.display.set_caption(
        f"{clock.get_fps():.0f} FPS | {PLAYER_NAME} #{net.my_id} | "
        f"{status} | {len(remotes)} others")
    pygame.display.flip()
    clock.tick(60)

net.close()
